In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../part1/cleaned_data.csv")

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (1338, 12)


,age,sex,bmi,children,smoker,region,charges,blood_pressure,exercise_frequency,pre_existing_condition,occupation_risk,annual_income
0,19,female,27.900,0,yes,southwest,16884.92400,139.0,Never,True,high,104158.67
1,18,male,33.770,1,no,southeast,1725.55230,129.9,Weekly,True,moderate,43530.88
2,28,male,33.000,3,no,southeast,4449.46200,111.1,Rarely,False,high,113004.75
3,33,male,22.705,0,no,northwest,21984.47061,126.9,Rarely,False,high,185041.26
4,32,male,28.880,0,no,northwest,3866.85520,134.7,Rarely,True,low,46747.97


In [2]:
y_reg = df["charges"]
y_clf = (df["charges"] > df["charges"].median()).astype(int)
X = df.drop(columns=["charges"])

In [3]:
exercise_mapping = {"Never": 0,"Rarely": 1,"Weekly": 2,"Daily": 3}
occupation_mapping = {"low": 0,"moderate": 1,"high": 2}
X["exercise_frequency"] = X["exercise_frequency"].map(exercise_mapping)
X["occupation_risk"] = X["occupation_risk"].map(occupation_mapping)
X = pd.get_dummies(X,columns=["sex", "smoker", "region"],drop_first=True)

In [4]:
X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Shape :", X_train_scaled.shape)
print("Testing Shape  :", X_test_scaled.shape)
print("\nTraining Labels")
print(y_clf_train.value_counts())

Training Shape : (1070, 13)
Testing Shape  : (268, 13)

Training Labels
1    547
0    523
Name: charges, dtype: int64


Part 3 - Advanced Modeling - Ensembles, Tuning, and Full ML Pipeline

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train_scaled,y_clf_train)

train_pred = dt_baseline.predict(X_train_scaled)
test_pred = dt_baseline.predict(X_test_scaled)


In [6]:
train_accuracy = accuracy_score(y_clf_train,train_pred)

test_accuracy = accuracy_score(y_clf_test,test_pred)

print("Decision tree baseline")
print(f"Training accuracy : {train_accuracy:.4f}")
print(f"Testing accuracy  : {test_accuracy:.4f}")

Decision tree baseline
Training accuracy : 1.0000
Testing accuracy  : 0.8507


In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_controlled = DecisionTreeClassifier(max_depth=5,min_samples_split=20,random_state=42)
dt_controlled.fit(X_train_scaled, y_clf_train)

DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)

In [8]:
train_pred_controlled = dt_controlled.predict(X_train_scaled)
test_pred_controlled = dt_controlled.predict(X_test_scaled)

In [9]:
train_acc_controlled = accuracy_score(y_clf_train,train_pred_controlled)
test_acc_controlled = accuracy_score(y_clf_test,test_pred_controlled)

print("Controlled decision tree")
print(f"Training accuracy: {train_acc_controlled:.4f}")
print(f"Testing accuracy: {test_acc_controlled:.4f}")

Controlled decision tree
Training accuracy: 0.9299
Testing accuracy: 0.9328


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_gini = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

dt_gini.fit(X_train_scaled, y_clf_train)
gini_pred = dt_gini.predict(X_test_scaled)
gini_accuracy = accuracy_score(y_clf_test, gini_pred)

In [11]:
dt_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

dt_entropy.fit(X_train_scaled, y_clf_train)
entropy_pred = dt_entropy.predict(X_test_scaled)
entropy_accuracy = accuracy_score(y_clf_test, entropy_pred)

In [12]:
print(f"Gini test accuracy: {gini_accuracy:.4f}")
print(f"Entropy test accuracy: {entropy_accuracy:.4f}")

Gini test accuracy: 0.9216
Entropy test accuracy: 0.9216


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

rf_model = RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42)
rf_model.fit(X_train_scaled, y_clf_train)

train_pred_rf = rf_model.predict(X_train_scaled)
test_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

train_acc_rf = accuracy_score(y_clf_train, train_pred_rf)
test_acc_rf = accuracy_score(y_clf_test, test_pred_rf)
auc_rf = roc_auc_score(y_clf_test, y_prob_rf)

print("Random forest performance")
print(f"Training accuracy: {train_acc_rf:.4f}")
print(f"Testing accuracy: {test_acc_rf:.4f}")
print(f"ROC-AUC: {auc_rf:.4f}")


Random forest performance
Training accuracy: 0.9617
Testing accuracy: 0.9366
ROC-AUC: 0.9555


In [14]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

top5_features = feature_importance.sort_values(by="Importance",ascending=False).head(5)
print("Top 5 Important Features")
display(top5_features)

Top 5 Important Features


,Feature,Importance
0,age,0.440912
9,smoker_yes,0.298966
1,bmi,0.059316
7,annual_income,0.057503
3,blood_pressure,0.054408


In [15]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_model.fit(X_train_scaled, y_clf_train)

train_pred_gb = gb_model.predict(X_train_scaled)
test_pred_gb = gb_model.predict(X_test_scaled)
y_prob_gb = gb_model.predict_proba(X_test_scaled)[:, 1]

train_acc_gb = accuracy_score(y_clf_train, train_pred_gb)
test_acc_gb = accuracy_score(y_clf_test, test_pred_gb)
auc_gb = roc_auc_score(y_clf_test, y_prob_gb)
print("Gradient boosting performance")
print(f"Training accuracy: {train_acc_gb:.4f}")
print(f"Testing accuracy: {test_acc_gb:.4f}")
print(f"ROC-AUC: {auc_gb:.4f}")

Gradient boosting performance
Training accuracy: 0.9589
Testing accuracy: 0.9366
ROC-AUC: 0.9633


In [16]:
least5 = feature_importance.sort_values(by="Importance").head(5)
print("Five least important features")
display(least5)

Five least important features


,Feature,Importance
10,region_northwest,0.004927
11,region_southeast,0.005432
12,region_southwest,0.005830
5,pre_existing_condition,0.007457
8,sex_male,0.009053


In [17]:
remove_features = least5["Feature"].tolist()
print("Removed features")
print(remove_features)

Removed features
['region_northwest', 'region_southeast', 'region_southwest', 'pre_existing_condition', 'sex_male']


In [18]:
X_train_reduced = pd.DataFrame(
    X_train_scaled,
    columns=X.columns
).drop(columns=remove_features)

X_test_reduced = pd.DataFrame(
    X_test_scaled,
    columns=X.columns
).drop(columns=remove_features)

In [19]:
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_reduced,y_clf_train)
full_auc = auc_rf

reduced_prob = rf_reduced.predict_proba(X_test_reduced)[:,1]
reduced_auc = roc_auc_score(y_clf_test,reduced_prob)

print("Full Model AUC:", round(full_auc,4))
print("Reduced Model AUC:", round(reduced_auc,4))


Full Model AUC: 0.9555
Reduced Model AUC: 0.96


In [20]:
comparison = pd.DataFrame({
    "Model":[
        "Random Forest (All Features)",
        "Random Forest (Reduced Features)"
    ],
    "ROC-AUC":[
        full_auc,
        reduced_auc
    ]
})

display(comparison)

,Model,ROC-AUC
0,Random Forest (All Features),0.955479
1,Random Forest (Reduced Features),0.959971


In [21]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

In [22]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)


In [23]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42)
    ),

    "Decision Tree (max_depth=5)": DecisionTreeClassifier(
        max_depth=5,min_samples_split=20,random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,max_depth=10,random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,learning_rate=0.1,max_depth=3,random_state=42
    )
}

In [24]:
results = []

for name, model in models.items():

    scores = cross_val_score(model,X,y_clf,cv=cv,scoring="roc_auc")
    results.append([name,scores.mean(),scores.std()])

cv_results = pd.DataFrame(results,
    columns=["Model","Mean AUC","Standard Deviation"]
)
display(cv_results)

,Model,Mean AUC,Standard Deviation
0,Logistic Regression,0.946606,0.016969
1,Decision Tree (max_depth=5),0.940877,0.016795
2,Random Forest,0.947917,0.017014
3,Gradient Boosting,0.950091,0.016608


In [25]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [26]:
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}
pipeline = make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),RandomForestClassifier(random_state=42))
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
grid_search = GridSearchCV(
    estimator=pipeline,param_grid=param_grid,cv=cv,scoring="roc_auc",n_jobs=-1
)

grid_search.fit(X_train, y_clf_train)
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(f"{grid_search.best_score_:.4f}")

Best parameters:
{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best Cross-Validation ROC-AUC:
0.9516


In [27]:
num_models = (
    len(param_grid["randomforestclassifier__n_estimators"])
    * len(param_grid["randomforestclassifier__max_depth"])
    * len(param_grid["randomforestclassifier__min_samples_leaf"])
)

total_fits = num_models*5

print(f"Model configurations: {num_models}")
print(f"Total model fits: {total_fits}")

Model configurations: 18
Total model fits: 90


In [28]:
from sklearn.metrics import roc_auc_score
import pandas as pd

best_pipeline = grid_search.best_estimator_
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
results = []

for f in fractions:

    n = int(f * len(X_train))
    X_subset = X_train.iloc[:n]
    y_subset = y_clf_train.iloc[:n]
    best_pipeline.fit(X_subset, y_subset)

    train_prob = best_pipeline.predict_proba(X_subset)[:, 1]
    train_auc = roc_auc_score(y_subset, train_prob)
    test_prob = best_pipeline.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_clf_test, test_prob)

    results.append([f, train_auc, test_auc])

learning_curve = pd.DataFrame(
    results,
    columns=["Training Fraction","Training AUC","Test AUC"]
)

display(learning_curve)

,Training Fraction,Training AUC,Test AUC
0,0.2,1.0,0.956827
1,0.4,1.0,0.952448
2,0.6,1.0,0.952392
3,0.8,1.0,0.959578
4,1.0,1.0,0.957164


In [29]:
import joblib

best_pipeline = grid_search.best_estimator_
joblib.dump(best_pipeline, "best_model.pkl")
print("Model saved successfully as best_model.pkl")

Model saved successfully as best_model.pkl


In [30]:
import joblib
import pandas as pd

loaded_model = joblib.load("best_model.pkl")
sample_data = pd.DataFrame([
    {
        "age": 25,
        "bmi": 22.5,
        "children": 0,
        "blood_pressure": 120,
        "exercise_frequency": 3,
        "pre_existing_condition": False,
        "occupation_risk": 0,
        "annual_income": 45000,
        "sex_male": 1,
        "smoker_yes": 0,
        "region_northwest": 0,
        "region_southeast": 0,
        "region_southwest": 1
    },
    {
        "age": 55,
        "bmi": 34.2,
        "children": 2,
        "blood_pressure": 145,
        "exercise_frequency": 1,
        "pre_existing_condition": True,
        "occupation_risk": 2,
        "annual_income": 85000,
        "sex_male": 0,
        "smoker_yes": 1,
        "region_northwest": 1,
        "region_southeast": 0,
        "region_southwest": 0
    }
])

predictions = loaded_model.predict(sample_data)
print("Predicted Classes:")
print(predictions)

Predicted Classes:
[0 1]
